# Practice Quiz · Lectures 14–17

**Suggested time: 45 minutes.** Complete A–F in the answer cells. Run the check cells after completing the corresponding question.

Place this notebook beside the original practice quiz. Run it with the folder containing `quiz_data` as your working directory. The only data files used are `quiz_data/fifa23_players_basic.csv`, `quiz_data/features.csv`, and `quiz_data/speed_mass.csv`.

A–C practice Pandas. D–F practice special methods, inheritance, and objects that contain other objects.

## A. Random assignment to three groups

1. Read `quiz_data/fifa23_players_basic.csv` into `df_players`. Store the number of rows in `n`.
2. Use `np.random.uniform` to create one random number per player between 0 and 1. Save the numbers in a new column called `draw`.
3. Use `.query()` to create these DataFrames:

   | DataFrame | Draw range |
   |---|---|
   | `group_a` | `draw < 0.25` |
   | `group_b` | `0.25 <= draw < 0.75` |
   | `group_c` | `draw >= 0.75` |

4. Print the percentage of all players in each group, rounded to two decimal places. Your results should be near 25%, 50%, and 25%.

In [48]:
# Your code here
import pandas as pd
import numpy as np
import random
df_players=pd.read_csv('quiz_data/fifa23_players_basic.csv')
n=len(df_players)
df_players["draw"]=np.random.uniform(0,1,n)
group_a=df_players.query("draw<0.25")
group_b=df_players.query("draw>=0.25 and draw<0.75")
group_c=df_players.query("draw>=0.75")
percent_a=len(group_a)/n
percent_b=len(group_b)/n
percent_c=len(group_c)/n
print(round(percent_a, 2), round(percent_b, 2), round(percent_c, 2))


0.25 0.5 0.25


In [3]:
# Check A
assert len(group_a) + len(group_b) + len(group_c) == n
assert (group_a["draw"] < 0.25).all()
assert ((group_b["draw"] >= 0.25) & (group_b["draw"] < 0.75)).all()
assert (group_c["draw"] >= 0.75).all()
print("Group checks passed.")

Group checks passed.


## B. Select, sort, and select again

Read `quiz_data/features.csv` into `df_cars`.

1. Use `.iloc` with a slice for both rows and columns to select cars at positions **36 through 45 inclusive** and the three adjacent columns `weight`, `acceleration`, and `vehicle_id`. Save the result as `df_subset`.
2. Sort `df_subset` by `weight` from lightest to heaviest and save the sorted table as `df_sorted`.
3. Use `.iloc` again to save the three lightest cars in `lightest_three`. Display the result. Keep all three selected columns.

In [49]:
# Your code here
df_cars=pd.read_csv('quiz_data/features.csv')
df_cars.head()
df_subset=df_cars.iloc[36:46,4:7]
df_sorted=df_subset.sort_values(by="weight",ascending=True)
lightest_three=df_sorted.iloc[:3,:]
display(lightest_three)


,weight,acceleration,vehicle_id
45,2962,13.5,A-1690275
37,3288,15.5,A-1690187
36,3302,15.5,F-1690176


In [13]:
# Check B
assert df_subset.shape == (10, 3)
assert list(df_subset.columns) == ["weight", "acceleration", "vehicle_id"]
assert list(df_subset.index) == list(range(36, 46))
assert df_sorted["weight"].is_monotonic_increasing
assert lightest_three.equals(df_sorted.head(3))
print("Selection and sorting checks passed.")

Selection and sorting checks passed.


## C. Convert speed and calculate momentum

Read `quiz_data/speed_mass.csv` into `df_vehicles`. Its speed and mass columns are called `"Speed (mph)"` and `"Mass (kg)"`.

1. Write a function `mph_to_mps(speed)` that converts a single speed from mph to m/s using $v_{m/s} = v_{mph} \times 1609.34 / 3600$.
2. Use `.apply()` on the speed column and store the returned Series in `speed_mps`.
3. Add a column named exactly `"Momentum (kg m/s)"`. Calculate momentum as mass in kilograms times speed in meters per second. Display the first five rows.

In [51]:
# Your code here
df_vehicles=pd.read_csv('quiz_data/speed_mass.csv')
def mph_to_mps(speed):
    return speed*1609.34/3600
df_vehicles.head()
speed_mps = df_vehicles["Speed (mph)"].apply(mph_to_mps)
df_vehicles["Momentum (kg m/s)"]=df_vehicles["Mass (kg)"]*speed_mps
display(df_vehicles.head(5))


,Vehicle ID,Speed (mph),Mass (kg),Momentum (kg m/s)
0,QA22,42.5,1441.67,27390.508585
1,TR80,77.0,1577.65,54305.859535
2,KA25,63.9,2276.71,65036.008367
3,AF65,55.9,1475.38,36868.969986
4,FP80,29.4,1254.24,16484.405246


In [18]:
# Check C
from math import isclose

assert isclose(mph_to_mps(60), 26.82233333333333)
first = df_vehicles.iloc[0]
assert isclose(
    first["Momentum (kg m/s)"],
    first["Mass (kg)"] * mph_to_mps(first["Speed (mph)"])
)
print("Conversion and momentum checks passed.")

Conversion and momentum checks passed.


## D. Make subtraction work for Fraction objects

Complete `__sub__(self, other)` in the supplied class so that `f1 - f2` returns a **new, simplified** `Fraction`. The original operands should keep their original numerators and denominators.

Use $a/b - c/d = (ad-bc)/(bd)$. The provided `reduce()` updates the fraction it is called on and returns `None`. Assume all input denominators are positive.

In [23]:
def gcd(m, n):
    while m % n != 0:
        m, n = n, m % n
    return n


class Fraction:
    def __init__(self, num, den):
        self.num = num
        self.den = den

    def __str__(self):
        return f"{self.num}/{self.den}"

    def reduce(self):
        common = gcd(self.num, self.den)
        self.num //= common
        self.den //= common

    def __sub__(self, other):
        new_num = self.num * other.den - other.num * self.den
        new_den = self.den * other.den
        result = Fraction(new_num, new_den)
        result.reduce()
        return result
        

In [24]:
# Check D
left = Fraction(5, 6)
right = Fraction(1, 3)
print(left - right)  # 1/2
print(left, right)   # 5/6 1/3
print(right - left)  # -1/2

1/2
5/6 1/3
-1/2


## E. Inheritance and a shared bonus

The base class below represents a student whose `rate()` method returns another student's lab skill. Write a child class `StudyGroupStudent` that meets these requirements:

- Inherit from `Student`. Set the class variable `bonus` to `2`.
- Override `rate(self, other)`. When `other` is also a `StudyGroupStudent`, return the other student's skill plus the shared bonus. For any ordinary `Student`, return their skill.
- Add a class method `set_bonus(cls, value)`. Use an assertion to accept values from 0 through 5 inclusive, and update the shared bonus.

The bonus affects the rating only. The underlying skill is stored by the parent class.

In [25]:
class Student:
    def __init__(self, name, lab_skill):
        self.name = name
        self.lab_skill = lab_skill

    def get_lab_skill(self):
        return self.lab_skill

    def rate(self, other):
        return other.get_lab_skill()

In [26]:
# Write StudyGroupStudent here
class StudyGroupStudent(Student):
    bonus=2
   
    def rate(self, other):
        rating = super().rate(other)
        if isinstance(other, StudyGroupStudent):
            rating += self.bonus
        return rating
    @classmethod
    def set_bonus(cls, new_bonus):
        assert 5>=new_bonus >= 0, "bonus must be between 0 and 5"
        cls.bonus = new_bonus


In [27]:
# Check E
StudyGroupStudent.set_bonus(2)
ana = StudyGroupStudent("Ana", 7)
ben = StudyGroupStudent("Ben", 6)
cy = Student("Cy", 8)

print(ana.rate(ben))  # 8
print(ana.rate(cy))   # 8
print(cy.rate(ben))   # 6

StudyGroupStudent.set_bonus(4)
print(ana.rate(ben))       # 10
print(ben.get_lab_skill()) # 6

try:
    StudyGroupStudent.set_bonus(6)
except AssertionError:
    print("Invalid bonus rejected.")
else:
    raise AssertionError("The setter should reject 6.")

8
8
6
10
6
Invalid bonus rejected.


## F. Find the first mutually acceptable lab partner

The supplied `LabStudent` class stores a name, a skill score, and the minimum skill the student will accept in a partner. Write a class called `LabSection`.

1. Its constructor takes a dictionary `roster` that maps a name to a tuple `(skill, minimum)`. Create a list called `self.students` containing one new `LabStudent` for each entry, in dictionary order.
2. Write `match_partners(self)`. Process students in list order. For each unpaired student, search the list from the beginning for the **first** candidate who meets all these conditions:
   - The candidate is a different object and is still unpaired.
   - The candidate's skill is at least the current student's minimum.
   - The current student's skill is at least the candidate's minimum.
3. Call `set_partner()` to pair them, then stop searching for that student. Leave a student unpaired if no candidate qualifies.

Hint: Both students must accept each other. A skill equal to a minimum qualifies.

In [28]:
class LabStudent:
    def __init__(self, name, skill, minimum):
        self.name = name
        self.skill = skill
        self.minimum = minimum
        self.partner = None

    def get_partner(self):
        return self.partner

    def set_partner(self, other):
        self.partner = other
        other.partner = self

In [46]:
# Write LabSection here
class LabSection:
    def __init__(self, roster):
        self.students = []
        for name in roster:
            skill, minimum = roster[name]
            self.students.append(LabStudent(name, skill, minimum))
    def match_partners(self):
        for student in self.students:
            if student.get_partner() is None:
             for potential_partner in self.students:
                if potential_partner.get_partner() is None and potential_partner.name != student.name:
                       
                    if potential_partner.skill >= student.minimum and student.skill>=potential_partner.minimum:
                            student.set_partner(potential_partner)
                            break
    

In [47]:
# Check F
roster = {
    "Ana": (7, 6),
    "Ben": (9, 8),
    "Cy": (6, 7),
    "Dee": (8, 8),
    "Eva": (7, 9),
    "Finn": (9, 8),
}

section = LabSection(roster)
section.match_partners()

for student in section.students:
    partner = student.get_partner()
    partner_name = None if partner is None else partner.name
    print(student.name, "->", partner_name)

# Ana -> Cy
# Ben -> Dee
# Cy -> Ana
# Dee -> Ben
# Eva -> None
# Finn -> None

Ana -> Cy
Ben -> Dee
Cy -> Ana
Dee -> Ben
Eva -> None
Finn -> None
